In [ ]:
"""
SESR-Based Flash Drought Detection Pipeline
============================================
A generalized implementation of the three-criterion flash drought
identification framework developed by Christian et al. (2019, 2022, 2023).

This pipeline detects flash drought events in any terrestrial study unit
using gridded reanalysis data and optional satellite vegetation indices.

References
----------
Christian, J.I., Basara, J.B., Otkin, J.A., Hunt, E.D., Wakefield, R.A.,
    Flanagan, P.X., & Xiao, X. (2019). A methodology for flash drought
    identification: Application of flash drought frequency across the
    United States. Journal of Hydrometeorology, 20(5), 833-846.

Christian, J.I., Basara, J.B., Hunt, E.D., Otkin, J.A., & Furtado, J.C.
    (2022). Flash drought development and cascading impacts associated with
    the 2010 Russian heatwave. Environmental Research Letters, 17(1), 014008.

Christian, J.I., et al. (2023). Global distribution, trends, and drivers
    of flash drought. Science, 380(6641), 168-173.

Usage
-----
1. Edit the CONFIGURATION block below to point to your data files
   and define your study units (polygons, points, or bounding boxes).
2. Run: python flash_drought_sesr_pipeline.py
3. Outputs are written to the directory specified in OUTPUT_DIR.

Requirements
------------
Python >= 3.8
numpy, pandas, scipy, netCDF4, geopandas, shapely

Author
------
[Ong'ondo et al., 2026] — generalized from study-specific implementation
License: MIT
"""

import os
import time
import numpy as np
import pandas as pd
import netCDF4 as nc4
import geopandas as gpd
from shapely.geometry import box
from scipy.interpolate import RegularGridInterpolator
from scipy.signal import savgol_filter

# CONFIGURATION — edit this block for your study

# --- Paths to input NetCDF files ---
ET_FILE   = "path/to/et_reanalysis.nc"        
PET_FILE  = "path/to/pet_reanalysis.nc"        
RZSM_FILE = "path/to/rzsm_reanalysis.nc"       

#Variable names inside the NetCDF files ---
ET_VAR   = "et"       
PET_VAR  = "pet"      
RZSM_VAR = "rzsm"     

# Study unit shapefiles
# Dictionary: {unit_id: path_to_shapefile}
# Each shapefile should contain the polygon(s) of one study unit.
STUDY_UNITS = {
    "unit_01": "path/to/shapefiles/unit_01.shp",
    "unit_02": "path/to/shapefiles/unit_02.shp",
    # add as many units as needed
}

# Temporal settings ---
STUDY_YEARS     = np.arange(1980, 2015)   
SEASON_START_DOY = 91                      
SEASON_END_DOY   = 304                     

# Flash drought detection thresholds (Christian et al. 2019) 
C1_PERCENTILE  = 25   
C2_MIN_PENTADS = 5    
C3_PERCENTILE  = 20  

# Savitzky-Golay smoothing parameters ---
SG_WINDOW_HALF = 10   
SG_POLYORDER   = 4    

# RZSM regridding 
# Set to True if your RZSM data is on a coarser grid than ET/PET
# and needs bilinear interpolation before extraction.
REGRID_RZSM    = True
RZSM_LON_0_360 = True 

# RZSM pentad index range (0-based) within the full year 
# Only needed if RZSM file contains full-year pentads.
# April-October in a 73-pentad year = pentads 18-60 (0-based)
RZSM_PENTAD_START = 18
RZSM_PENTAD_END   = 60

# Output 
OUTPUT_DIR = "flash_drought_output"


# CONSTANTS 

PENTAD_DAYS = 5
DECADE      = 10.0

# Days in growing season
SEASON_DAYS   = SEASON_END_DOY - SEASON_START_DOY + 1
N_PENTADS     = SEASON_DAYS // PENTAD_DAYS          
N_YEARS       = len(STUDY_YEARS)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# SECTION 1 — DATA LOADING


def load_netcdf(fpath, varname):
    """
    Load a NetCDF variable along with lon, lat, and year dimensions.

    Parameters
    ----------
    fpath   : str — path to NetCDF file
    varname : str — name of the variable to load

    Returns
    -------
    lon  : 1-D array of longitudes
    lat  : 1-D array of latitudes
    years: 1-D int array of years
    data : 4-D array (lon, lat, time_within_year, year)
    """
    if not os.path.exists(fpath):
        raise FileNotFoundError(f"File not found: {fpath}")
    ds   = nc4.Dataset(fpath, "r")
    lon  = np.array(ds.variables["lon"][:])
    lat  = np.array(ds.variables["lat"][:])
    yrs  = np.array(ds.variables["year"][:]).astype(int)
    data = np.array(ds.variables[varname][:])
    ds.close()
    print(f"  Loaded {os.path.basename(fpath)}  shape={data.shape}")
    return lon, lat, yrs, data


def subset_years(data, years_in_file, study_years):
    """Subset a data array to only the required study years."""
    mask = np.isin(years_in_file, study_years)
    return data[:, :, :, mask], years_in_file[mask]


def regrid_rzsm(rzsm_raw, lon_sm_raw, lat_sm, lon_target, lat_target,
                convert_lon=True):
    """
    Bilinear regrid coarse RZSM (e.g. 2.5 deg) to a finer grid (e.g. 0.625 deg).

    Parameters
    ----------
    rzsm_raw    : array (lon, lat, pentad, year) on coarse grid
    lon_sm_raw  : coarse longitude array (may be 0-360)
    lat_sm      : coarse latitude array
    lon_target  : fine longitude array (-180 to 180)
    lat_target  : fine latitude array
    convert_lon : bool — convert 0-360 to -180-180 if True

    Returns
    -------
    rzsm_fine : array (len(lon_target), len(lat_target), n_pentads, n_years)
    """
    lon_sm = lon_sm_raw.copy()
    if convert_lon:
        lon_sm[lon_sm > 180] -= 360
        idx    = np.argsort(lon_sm)
        lon_sm = lon_sm[idx]
        rzsm_raw = rzsm_raw[idx]

    n_p, n_y = rzsm_raw.shape[2], rzsm_raw.shape[3]
    out = np.full(
        (len(lon_target), len(lat_target), n_p, n_y),
        np.nan, dtype=np.float32
    )
    t0 = time.time()
    for y_i in range(n_y):
        for p_i in range(n_p):
            sl = np.clip(
                rzsm_raw[:, :, p_i, y_i].astype(np.float64), 0.0, 1.5
            )
            interp = RegularGridInterpolator(
                (lon_sm, lat_sm), sl,
                method='linear', bounds_error=False, fill_value=np.nan
            )
            lg, latg = np.meshgrid(lon_target, lat_target, indexing='ij')
            pts = np.column_stack([lg.ravel(), latg.ravel()])
            out[:, :, p_i, y_i] = (
                interp(pts).reshape(len(lon_target), len(lat_target))
                .astype(np.float32)
            )
        if (y_i + 1) % 5 == 0:
            print(f"    Regrid: year {y_i+1}/{n_y} ({time.time()-t0:.0f}s)")
    return out


# SECTION 2 — SPATIAL EXTRACTION


def get_overlap_weights(shp_path, lon, lat):
    """
    Compute area-overlap weights between gridded pixels and a study unit polygon.

    Each pixel's weight = (intersection area with polygon) / (total overlap area).
    Falls back to nearest-centroid if no pixel intersects the polygon.

    Parameters
    ----------
    shp_path : str — path to shapefile defining the study unit boundary
    lon      : 1-D array of grid longitudes
    lat      : 1-D array of grid latitudes

    Returns
    -------
    lat_idx  : array of latitude indices for overlapping pixels
    lon_idx  : array of longitude indices for overlapping pixels
    weights  : normalized area-overlap weights (sum = 1)
    n_pixels : int — number of overlapping pixels
    method   : str — 'overlap' or 'nearest' (fallback)
    """
    gdf      = gpd.read_file(shp_path).to_crs("EPSG:4326")
    polygon  = (gdf.union_all() if hasattr(gdf, 'union_all')
                else gdf.unary_union)
    bounds   = polygon.bounds

    dlon_h = abs(lon[1] - lon[0]) / 2 if len(lon) > 1 else 0.3125
    dlat_h = abs(lat[1] - lat[0]) / 2 if len(lat) > 1 else 0.25

    lon_cand = np.where(
        (lon >= bounds[0] - dlon_h*2) & (lon <= bounds[2] + dlon_h*2)
    )[0]
    lat_cand = np.where(
        (lat >= bounds[1] - dlat_h*2) & (lat <= bounds[3] + dlat_h*2)
    )[0]

    li_list, loi_list, wt_list = [], [], []

    for lai in lat_cand:
        for loi in lon_cand:
            px = box(
                lon[loi] - dlon_h, lat[lai] - dlat_h,
                lon[loi] + dlon_h, lat[lai] + dlat_h
            )
            if not px.intersects(polygon):
                continue
            isect = px.intersection(polygon)
            if isect.is_empty:
                continue
            area = (gpd.GeoDataFrame(geometry=[isect], crs="EPSG:4326")
                    .to_crs("EPSG:6933").geometry.area.values[0])
            if area > 0:
                li_list.append(lai)
                loi_list.append(loi)
                wt_list.append(area)

    if not li_list:
        # Fallback: nearest grid cell to polygon centroid
        cx  = polygon.centroid.x
        cy  = polygon.centroid.y
        loi = int(np.argmin(np.abs(lon - cx)))
        lai = int(np.argmin(np.abs(lat - cy)))
        return np.array([lai]), np.array([loi]), np.array([1.0]), 1, 'nearest'

    wt = np.array(wt_list, dtype=np.float64)
    wt /= wt.sum()
    return (np.array(li_list), np.array(loi_list),
            wt, len(li_list), 'overlap')


def weighted_mean(data4d, lat_idx, lon_idx, weights):
    """
    Compute area-overlap weighted mean across overlapping pixels.

    Parameters
    ----------
    data4d  : array (lon, lat, time, year)
    lat_idx : array of latitude indices
    lon_idx : array of longitude indices
    weights : normalized weight for each pixel

    Returns
    -------
    array (time, year) — weighted mean time series
    """
    pixels = np.stack(
        [data4d[lon_idx[i], lat_idx[i], :, :]
         for i in range(len(lat_idx))],
        axis=0
    )
    return np.nansum(pixels * weights[:, None, None], axis=0)

# SECTION 3 — ESR / SESR PROCESSING


def daily_to_pentads(daily_2d, n_pentads, pentad_days=5):
    """
    Aggregate daily time series to pentad means.

    Parameters
    ----------
    daily_2d  : array (n_days, n_years)
    n_pentads : int — number of pentads in the season
    pentad_days : int — days per pentad (default 5)

    Returns
    -------
    array (n_pentads, n_years)
    """
    n_use = n_pentads * pentad_days
    return (daily_2d[:n_use, :]
            .reshape(n_pentads, pentad_days, daily_2d.shape[1])
            .mean(axis=1))


def standardize_per_pentad(arr2d):
    """
    Z-score standardization per pentad using full-period climatology.

    Removes the seasonal cycle by normalizing each pentad to its own
    long-term mean and standard deviation across all years.

    Parameters
    ----------
    arr2d : array (n_pentads, n_years)

    Returns
    -------
    z     : standardized array (n_pentads, n_years)
    mu    : per-pentad mean (n_pentads,)
    sigma : per-pentad std  (n_pentads,)
    """
    mu    = np.nanmean(arr2d, axis=1, keepdims=True)
    sigma = np.nanstd( arr2d, axis=1, keepdims=True, ddof=1)
    sigma[sigma == 0] = np.nan
    z = (arr2d - mu) / sigma
    return z, mu.squeeze(), sigma.squeeze()


def savgol_smooth_pentads(arr2d, window_half, polyorder):
    """
    Apply Savitzky-Golay filter to each year's pentad time series.

    Parameters
    ----------
    arr2d       : array (n_pentads, n_years)
    window_half : int — half-window length in pentads
    polyorder   : int — polynomial degree

    Returns
    -------
    array (n_pentads, n_years) — smoothed
    """
    window = 2 * window_half + 1
    out    = np.full_like(arr2d, np.nan)
    for yi in range(arr2d.shape[1]):
        col   = arr2d[:, yi]
        valid = int(np.sum(~np.isnan(col)))
        out[:, yi] = (
            savgol_filter(col, window_length=window, polyorder=polyorder)
            if valid >= window else col
        )
    return out


def compute_delta_sesr(sesr_smooth):
    """
    Compute ΔSESRz: first difference of smoothed SESR, standardized per pentad.

    Parameters
    ----------
    sesr_smooth : array (n_pentads, n_years) — SG-smoothed SESR

    Returns
    -------
    dsesr_raw : array (n_pentads-1, n_years) — raw first differences
    dsesr_z   : array (n_pentads-1, n_years) — standardized first differences
    """
    dsesr_raw       = np.diff(sesr_smooth, axis=0)
    dsesr_z, _, _   = standardize_per_pentad(dsesr_raw)
    return dsesr_raw, dsesr_z


def pentad_percentile(arr2d, percentile):
    """
    Compute per-pentad percentile threshold across all years.

    Parameters
    ----------
    arr2d      : array (n_pentads, n_years)
    percentile : float — percentile value (0-100)

    Returns
    -------
    array (n_pentads,) — threshold value per pentad
    """
    return np.nanpercentile(arr2d, percentile, axis=1)


# SECTION 4 — FLASH DROUGHT IDENTIFICATION


def identify_flash_droughts(dsesr_z, srzsm, c1_threshold, c3_threshold,
                            esr_pentads, study_years,
                            c1_pctile=25, c2_min=5, c3_pctile=20):
    """
    Three-criterion flash drought identification (Christian et al. 2019).

    C1: ΔSESRz <= per-pentad 25th percentile (rapid evaporative decline)
    C2: >= 5 consecutive pentads meeting C1  (sustained ≥ 30-day intensification)
    C3: SRZSM at event end <= per-pentad 20th percentile (soil moisture deficit)

    Parameters
    ----------
    dsesr_z     : array (n_diffs, n_years) — standardized ΔSESRz
    srzsm       : array (n_pentads, n_years) — standardized RZSM
                  pass None to skip C3 (all C1+C2 events become confirmed)
    c1_threshold: array (n_diffs,) — per-pentad C1 cutoffs
    c3_threshold: array (n_pentads,) — per-pentad C3 cutoffs
    esr_pentads : array (n_pentads, n_years) — raw ESR for deficit calculation
    study_years : 1-D int array of years
    c1_pctile   : float — C1 percentile (default 25)
    c2_min      : int   — minimum consecutive pentads for C2 (default 5)
    c3_pctile   : float — C3 percentile (default 20)

    Returns
    -------
    confirmed  : list of dicts — events meeting all three criteria
    candidates : list of dicts — events meeting C1+C2 only
    """
    n_dp      = dsesr_z.shape[0]
    n_years   = len(study_years)
    confirmed  = []
    candidates = []

    for yi in range(n_years):
        yr    = int(study_years[yi])
        d_col = dsesr_z    [:, yi]
        e_col = esr_pentads[:, yi]
        g_col = srzsm      [:, yi] if srzsm is not None else None
        c1    = d_col <= c1_threshold

        p = 0
        while p < n_dp:
            if c1[p]:
                run_start = p
                while p < n_dp and c1[p]:
                    p += 1
                run_changes = p - run_start

                if run_changes >= c2_min:
                    fp = min(run_start + run_changes, esr_pentads.shape[0] - 1)
                    p0 = run_start
                    p1 = min(p0 + run_changes + 1, esr_pentads.shape[0])
                    cumdef = float(np.nansum(1.0 - e_col[p0:p1]))

                    # C3 check
                    if srzsm is not None:
                        gw_val        = float(g_col[fp])
                        gw_thr        = float(c3_threshold[fp])
                        confirmed_flag = gw_val <= gw_thr
                    else:
                        gw_val        = np.nan
                        gw_thr        = np.nan
                        confirmed_flag = True   # no C3 → all C1+C2 confirmed

                    ev = dict(
                        year               = yr,
                        onset_pentad       = run_start + 1,
                        end_pentad         = run_start + run_changes,
                        duration_pentads   = run_changes + 1,
                        duration_days      = (run_changes + 1) * PENTAD_DAYS,
                        peak_dsesr_z       = float(
                            np.nanmin(d_col[run_start:run_start+run_changes])),
                        final_pentad       = fp + 1,
                        srzsm_final        = gw_val,
                        srzsm_threshold    = gw_thr,
                        cumulative_deficit = cumdef,
                        c3_met             = confirmed_flag,
                    )
                    candidates.append(ev)
                    if confirmed_flag:
                        confirmed.append(ev)
            else:
                p += 1

    return confirmed, candidates


# SECTION 5 — MAIN PIPELINE


def run_pipeline():
    print("=" * 65)
    print("  SESR Flash Drought Detection Pipeline")
    print("  Christian et al. (2019, 2022, 2023) framework")
    print("=" * 65)

    # Load ET and PET 
    print("\n[1] Loading ET and PET...")
    lon, lat, years_et, ET_raw  = load_netcdf(ET_FILE,  ET_VAR)
    _,   _,   years_pet, PET_raw = load_netcdf(PET_FILE, PET_VAR)

    ET_all,  years = subset_years(ET_raw,  years_et,  STUDY_YEARS)
    PET_all, _     = subset_years(PET_raw, years_pet, STUDY_YEARS)
    n_years        = len(years)
    print(f"  Grid: {len(lon)} lon x {len(lat)} lat  |  "
          f"Years: {years[0]}–{years[-1]}  (n={n_years})")

    # Load and optionally regrid RZSM
    RZSM_all = None
    if RZSM_VAR is not None:
        print("\n[2] Loading RZSM...")
        lon_sm, lat_sm, years_sm, rzsm_raw = load_netcdf(RZSM_FILE, RZSM_VAR)

        # Subset to growing season pentads if needed
        if rzsm_raw.ndim == 4 and rzsm_raw.shape[2] > N_PENTADS:
            rzsm_raw = rzsm_raw[:, :,
                                 RZSM_PENTAD_START:RZSM_PENTAD_END + 1, :]
        rzsm_raw, _ = subset_years(rzsm_raw, years_sm, STUDY_YEARS)

        if REGRID_RZSM:
            print("  Regridding to ET/PET grid...")
            RZSM_all = regrid_rzsm(rzsm_raw, lon_sm, lat_sm,
                                   lon, lat, RZSM_LON_0_360)
        else:
            RZSM_all = rzsm_raw
    else:
        print("\n[2] RZSM not configured — C3 will be skipped.")

    #Loop over study units
    print(f"\n[3] Processing {len(STUDY_UNITS)} study unit(s)...\n")

    all_events    = []
    all_summaries = []

    for unit_id, shp_path in STUDY_UNITS.items():
        print(f"{'─'*65}")
        print(f"  Unit: {unit_id}")

        if not os.path.exists(shp_path):
            print(f"  WARNING: Shapefile not found — {shp_path}  (skipping)")
            continue

        # Pixel extraction
        lat_idx, lon_idx, weights, n_px, method = \
            get_overlap_weights(shp_path, lon, lat)
        print(f"  Pixels: {n_px}  |  Method: {method}")

        # Weighted area means
        et_day   = weighted_mean(ET_all,  lat_idx, lon_idx, weights)
        pet_day  = weighted_mean(PET_all, lat_idx, lon_idx, weights)

        # ESR: clip to [0, 1], aggregate to pentads
        with np.errstate(invalid="ignore", divide="ignore"):
            esr_day = np.where(pet_day > 0.01, et_day / pet_day, np.nan)
        esr_day  = np.clip(esr_day, 0.0, 1.0)
        esr_pen  = daily_to_pentads(esr_day, N_PENTADS)

        # SESR: standardize, smooth, difference
        sesr, _, _     = standardize_per_pentad(esr_pen)
        sesr_sm        = savgol_smooth_pentads(sesr, SG_WINDOW_HALF, SG_POLYORDER)
        dsesr_raw, dsesr_z = compute_delta_sesr(sesr_sm)

        # RZSM: extract and standardize for C3
        srzsm      = None
        c3_thr     = None
        if RZSM_all is not None:
            rzsm_pen        = weighted_mean(RZSM_all, lat_idx, lon_idx, weights)
            srzsm, _, _     = standardize_per_pentad(rzsm_pen)
            c3_thr          = pentad_percentile(srzsm, C3_PERCENTILE)

        # Percentile thresholds
        c1_thr = pentad_percentile(dsesr_z, C1_PERCENTILE)

        # Flash drought identification
        confirmed, candidates = identify_flash_droughts(
            dsesr_z, srzsm, c1_thr, c3_thr, esr_pen,
            years, C1_PERCENTILE, C2_MIN_PENTADS, C3_PERCENTILE
        )
        freq_per_decade = len(confirmed) / (n_years / DECADE)

        print(f"  C1+C2 candidates:   {len(candidates)}")
        print(f"  Confirmed events:   {len(confirmed)}")
        print(f"  Frequency:          {freq_per_decade:.2f} events/decade")

        # Tag events with unit_id and save
        for ev in confirmed:
            ev["unit_id"] = unit_id
            all_events.append(ev)
        for ev in candidates:
            ev["unit_id"] = unit_id

        if confirmed:
            pd.DataFrame(confirmed).to_csv(
                os.path.join(OUTPUT_DIR, f"{unit_id}_confirmed_events.csv"),
                index=False
            )

        # Save full time series
        rows = []
        for yi, yr in enumerate(years):
            for pi in range(N_PENTADS):
                row = {
                    "unit_id"     : unit_id,
                    "year"        : int(yr),
                    "pentad"      : pi + 1,
                    "esr"         : round(float(esr_pen [pi, yi]), 6),
                    "sesr"        : round(float(sesr    [pi, yi]), 6),
                    "sesr_smooth" : round(float(sesr_sm [pi, yi]), 6),
                }
                if srzsm is not None:
                    row["srzsm"] = round(float(srzsm[pi, yi]), 6)
                rows.append(row)
        pd.DataFrame(rows).to_csv(
            os.path.join(OUTPUT_DIR, f"{unit_id}_timeseries.csv"),
            index=False
        )

        # Summary statistics
        dur_days = [e["duration_days"] for e in confirmed]
        pk_dsesr = [e["peak_dsesr_z"]  for e in confirmed]
        cumdef   = [e["cumulative_deficit"] for e in confirmed]

        all_summaries.append({
            "unit_id"               : unit_id,
            "n_overlapping_pixels"  : n_px,
            "extraction_method"     : method,
            "n_confirmed_events"    : len(confirmed),
            "n_candidate_events"    : len(candidates),
            "freq_per_decade"       : round(freq_per_decade, 3),
            "mean_duration_days"    : round(float(np.mean(dur_days)), 1)
                                      if dur_days else np.nan,
            "mean_peak_dsesr_z"     : round(float(np.mean(pk_dsesr)), 3)
                                      if pk_dsesr else np.nan,
            "mean_cumulative_deficit": round(float(np.mean(cumdef)), 3)
                                       if cumdef else np.nan,
            "mean_seasonal_esr"     : round(float(np.nanmean(esr_pen)), 4),
        })

    # Cross-unit summary
    print(f"\n{'='*65}")
    print("  CROSS-UNIT SUMMARY")
    print(f"{'='*65}")

    df_summary = pd.DataFrame(all_summaries)
    print(df_summary[[
        "unit_id", "n_overlapping_pixels",
        "n_confirmed_events", "freq_per_decade", "mean_duration_days"
    ]].to_string(index=False))

    df_summary.to_csv(
        os.path.join(OUTPUT_DIR, "flash_drought_summary_all_units.csv"),
        index=False
    )

    if all_events:
        pd.DataFrame(all_events).to_csv(
            os.path.join(OUTPUT_DIR, "all_confirmed_events_combined.csv"),
            index=False
        )

    print(f"\n  Outputs written to: {OUTPUT_DIR}")
    print("  Pipeline complete.")
    return df_summary, all_events



# ENTRY POINT
if __name__ == "__main__":
    summary, events = run_pipeline()
